1. 將每一篇討論文章進行切割，以一個句子為單位去切
2. 使用BERTTopic來進行每一個句子的主題辨識，達到文本資料降維的功效。

In [5]:
import pandas as pd
import re

def semantic_content_split(df):
    """
    針對新聞文章內容進行語意切割
    
    Args:
        df: 包含content欄位的DataFrame
    
    Returns:
        processed_df: 切割後的DataFrame，保留原始資料的其他欄位
    """
    processed_rows = []
    total_chunks = 0
    
    for idx, row in df.iterrows():
        # 檢查content是否為空
        if pd.isna(row['content']):
            print(f"跳過空內容: ID {row['_id']}")
            continue
            
        content = row['content'].strip()
        if not content:
            print(f"跳過空字串內容: ID {row['_id']}")
            continue
            
        # 移除引號
        content = content.strip('"')
        
        # 1. 先用明顯的分隔符號切割
        # 包括段落符號、標題標記等
        paragraphs = re.split(r'\n+|。(?=[^。]*?[，。！？])|【.*?】', content)
        paragraphs = [p.strip() for p in paragraphs if p and p.strip()]
        
        current_chunk = []
        current_length = 0
        chunk_id = 0
        chunks_for_article = 0
        
        for para in paragraphs:
            # 如果單個段落就超過1000字
            if len(para) > 1000:
                # 強制按照字數切割
                for i in range(0, len(para), 800):  # 使用800作為切割點以留有餘裕
                    chunk = para[i:i+800]
                    new_row = row.copy()
                    new_row['content'] = chunk
                    new_row['chunk_id'] = f"{row['_id']}_{chunk_id}"
                    new_row['original_id'] = row['_id']
                    new_row['chunk_size'] = len(chunk)
                    processed_rows.append(new_row)
                    chunk_id += 1
                    chunks_for_article += 1
                current_chunk = []
                current_length = 0
                continue
                
            # 如果當前段落加上去會超過1000字
            if current_length + len(para) > 1000:
                if current_chunk:
                    # 保存當前chunk
                    new_row = row.copy()
                    chunk_content = '\n'.join(current_chunk)
                    new_row['content'] = chunk_content
                    new_row['chunk_id'] = f"{row['_id']}_{chunk_id}"
                    new_row['original_id'] = row['_id']
                    new_row['chunk_size'] = len(chunk_content)
                    processed_rows.append(new_row)
                    chunk_id += 1
                    chunks_for_article += 1
                    
                # 重置current_chunk
                current_chunk = [para]
                current_length = len(para)
            else:
                current_chunk.append(para)
                current_length += len(para)
        
        # 處理最後一個chunk
        if current_chunk:
            new_row = row.copy()
            chunk_content = '\n'.join(current_chunk)
            new_row['content'] = chunk_content
            new_row['chunk_id'] = f"{row['_id']}_{chunk_id}"
            new_row['original_id'] = row['_id']
            new_row['chunk_size'] = len(chunk_content)
            processed_rows.append(new_row)
            chunks_for_article += 1
        
        total_chunks += chunks_for_article
        if chunks_for_article > 1:
            print(f"文章 {row['_id']} 被切割成 {chunks_for_article} 個片段")
    
    # 創建新的DataFrame
    processed_df = pd.DataFrame(processed_rows)
    
    print(f"\n切割統計:")
    print(f"總共處理文章數: {len(df)}")
    print(f"總共產生片段數: {total_chunks}")
    print(f"平均每篇文章片段數: {total_chunks/len(df):.2f}")
    
    return processed_df

def process_csv_file(file_path):
    """
    處理CSV檔案的主函數
    
    Args:
        file_path: CSV檔案路徑
    """
    # 讀取CSV檔案
    df = pd.read_csv(file_path)
    
    # 進行內容切割
    processed_df = semantic_content_split(df)
    
    # 儲存處理後的結果
    output_path = file_path.replace('.csv', '_processed.csv')
    processed_df.to_csv(output_path, index=False)
    
    print(f"\n處理完成，結果已儲存至: {output_path}")
    
    return processed_df

# 使用範例
file_path = './data/Xiemen.csv'
processed_df = process_csv_file(file_path)


文章 66765f9ffb88ab4b048dd582 被切割成 5 個片段
文章 66765f9ffb88ab4b048dd583 被切割成 2 個片段
文章 66765f9ffb88ab4b048dd584 被切割成 5 個片段
文章 66765f9ffb88ab4b048dd586 被切割成 3 個片段
文章 66765f9ffb88ab4b048dd587 被切割成 2 個片段
文章 66765f9ffb88ab4b048dd588 被切割成 3 個片段
文章 66765f9ffb88ab4b048dd589 被切割成 2 個片段
文章 66765f9ffb88ab4b048dd58a 被切割成 4 個片段
文章 66765f9ffb88ab4b048dd58c 被切割成 2 個片段
文章 66765f9ffb88ab4b048dd58d 被切割成 2 個片段
文章 66765f9ffb88ab4b048dd58e 被切割成 2 個片段
文章 66765f9ffb88ab4b048dd591 被切割成 2 個片段
文章 66765f9ffb88ab4b048dd592 被切割成 2 個片段
文章 66765f9ffb88ab4b048dd593 被切割成 3 個片段
文章 66765f9ffb88ab4b048dd594 被切割成 2 個片段
文章 66765f9ffb88ab4b048dd595 被切割成 2 個片段
文章 66765f9ffb88ab4b048dd596 被切割成 2 個片段
文章 66765f9ffb88ab4b048dd597 被切割成 2 個片段
文章 66765f9ffb88ab4b048dd598 被切割成 2 個片段
文章 66765f9ffb88ab4b048dd59c 被切割成 2 個片段
文章 66765f9ffb88ab4b048dd59d 被切割成 2 個片段
文章 66765f9ffb88ab4b048dd59e 被切割成 2 個片段
文章 66765f9ffb88ab4b048dd59f 被切割成 2 個片段
文章 66765f9ffb88ab4b048dd5a1 被切割成 3 個片段
文章 66765f9ffb88ab4b048dd5a2 被切割成 3 個片段
文章 66765f9ffb88ab4b048dd5

In [9]:
processed_df.columns

Index(['_id', 'URL', 'title', 'year', 'month', 'day', 'date', 'content',
       'chunk_id', 'original_id', 'chunk_size'],
      dtype='object')

In [11]:
import re
import pandas as pd
from typing import List, Dict, Any
import logging

class ContentSplitter:
    """
    新聞文章內容切割器
    """
    
    def __init__(self, max_length: int = 1000, min_length: int = 100):
        """
        初始化切割器
        
        Args:
            max_length: 段落最大長度
            min_length: 段落最小長度
        """
        self.max_length = max_length
        self.min_length = min_length
        
        # 設置logging
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s'
        )
        self.logger = logging.getLogger(__name__)
        
    def clean_text(self, text: str) -> str:
        """
        清理文本
        
        Args:
            text: 原始文本
        
        Returns:
            清理後的文本
        """
        # 移除多餘空白
        text = re.sub(r'\s+', ' ', text)
        # 移除特殊符號
        text = re.sub(r'[^\w\s\u4e00-\u9fff。，、：""《》【】？！；]', '', text)
        # 統一標點符號
        text = text.replace('!', '！').replace('?', '？').replace(';', '；')
        return text.strip()
    
    def initial_split(self, text: str) -> List[str]:
        """
        初步切割文本
        
        Args:
            text: 清理後的文本
            
        Returns:
            初步切割後的段落列表
        """
        # 定義切割規則
        split_patterns = [
            # 多個換行
            r'\n\s*\n+',
            # 標題標記
            r'(?<=。)\s*(?=【.*?】)',
            # 數字編號
            r'(?<=。)\s*(?=\d+[、.]\s*\w)',
            # 中文編號
            r'(?<=。)\s*(?=第[一二三四五六七八九十]+[、，：])',
            # 引述
            r'(?<=。)\s*(?=[^，。！？]+[：說表示])',
            # 時間/日期
            r'(?<=。)\s*(?=\d{4}年|\d{1,2}月|\d{1,2}日)'
        ]
        
        pattern = '|'.join(split_patterns)
        paragraphs = re.split(pattern, text)
        
        # 清理空白
        paragraphs = [p.strip() for p in paragraphs if p and p.strip()]
        
        return paragraphs
    
    def merge_short_paragraphs(self, paragraphs: List[str]) -> List[str]:
        """
        合併過短的段落
        
        Args:
            paragraphs: 段落列表
            
        Returns:
            合併後的段落列表
        """
        result = []
        current = []
        current_length = 0
        
        for p in paragraphs:
            if len(p) < self.min_length:
                if current_length + len(p) <= self.max_length:
                    current.append(p)
                    current_length += len(p)
                else:
                    if current:
                        result.append(''.join(current))
                    current = [p]
                    current_length = len(p)
            else:
                if current:
                    result.append(''.join(current))
                result.append(p)
                current = []
                current_length = 0
                
        if current:
            result.append(''.join(current))
            
        return result
    
    def split_long_paragraph(self, paragraph: str) -> List[str]:
        """
        切割過長的段落
        
        Args:
            paragraph: 單個段落文本
            
        Returns:
            切割後的段落列表
        """
        if len(paragraph) <= self.max_length:
            return [paragraph]
            
        # 按句子切分
        sentences = re.split(r'([。！？])', paragraph)
        # 重組句子(保留標點)
        sentences = [''.join(i) for i in zip(sentences[0::2], sentences[1::2] + [''])]
        
        result = []
        current = []
        current_length = 0
        
        for sent in sentences:
            if len(sent) > self.max_length:
                # 如果單個句子超過最大長度,強制按字數切割
                if current:
                    result.append(''.join(current))
                
                # 強制切割
                for i in range(0, len(sent), self.max_length - 100):
                    chunk = sent[i:i + self.max_length - 100]
                    result.append(chunk)
                    
                current = []
                current_length = 0
                continue
                
            if current_length + len(sent) > self.max_length:
                result.append(''.join(current))
                current = [sent]
                current_length = len(sent)
            else:
                current.append(sent)
                current_length += len(sent)
                
        if current:
            result.append(''.join(current))
            
        return result
    
    def process_content(self, content: str) -> List[str]:
        """
        處理單篇文章內容
        
        Args:
            content: 文章內容
            
        Returns:
            處理後的段落列表
        """
        # 清理文本
        cleaned_text = self.clean_text(content)
        
        # 初步切割
        initial_paragraphs = self.initial_split(cleaned_text)
        
        # 處理過長和過短的段落
        result = []
        for p in initial_paragraphs:
            if len(p) > self.max_length:
                result.extend(self.split_long_paragraph(p))
            else:
                result.append(p)
                
        # 合併過短段落
        result = self.merge_short_paragraphs(result)
        
        return result
    
    def process_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        處理DataFrame中的內容
        
        Args:
            df: 包含content欄位的DataFrame
            
        Returns:
            處理後的DataFrame
        """
        processed_rows = []
        
        for idx, row in df.iterrows():
            if pd.isna(row['content']):
                self.logger.warning(f"跳過空內容: ID {row['_id']}")
                continue
                
            # 處理內容
            paragraphs = self.process_content(row['content'])
            
            # 為每個段落創建新行
            for i, para in enumerate(paragraphs):
                new_row = row.copy()
                new_row['content'] = para
                new_row['chunk_id'] = f"{row['_id']}_{i}"
                new_row['original_id'] = row['_id']
                new_row['chunk_size'] = len(para)
                processed_rows.append(new_row)
                
        # 創建新的DataFrame    
        processed_df = pd.DataFrame(processed_rows)
        
        self.logger.info(f"處理前文檔數: {len(df)}")
        self.logger.info(f"處理後文檔數: {len(processed_df)}")
        self.logger.info(f"平均chunk大小: {processed_df['chunk_size'].mean():.2f}")
        
        return processed_df

# 使用示例
if __name__ == "__main__":
    # 讀取數據
    df = pd.read_csv('./data/Xiemen.csv')
    
    # 初始化切割器
    splitter = ContentSplitter(max_length=1000, min_length=100)
    
    # 處理數據
    processed_df = splitter.process_dataframe(df)
    
    # 保存結果
    processed_df.to_csv('output.csv', index=False)
    print("處理完成!")


2024-12-02 17:03:24,786 - INFO - 處理前文檔數: 442
2024-12-02 17:03:24,786 - INFO - 處理後文檔數: 1562
2024-12-02 17:03:24,787 - INFO - 平均chunk大小: 342.01


處理完成!


#### 使用句號進行切割

In [1]:
import pandas as pd
import re
from typing import List
import logging

class SimpleSentenceSplitter:
    """
    簡單的句子切割器 - 以句號為切割依據
    """
    def __init__(self):
        # 設置logging
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s'
        )
        self.logger = logging.getLogger(__name__)
    
    def split_content(self, text: str) -> List[str]:
        """
        將文本按句號切割成句子
        
        Args:
            text: 原始文本
            
        Returns:
            切割後的句子列表
        """
        if pd.isna(text):
            return []
            
        # 使用正則表達式切割,保留句號
        sentences = re.split(r'(。)', text)
        
        # 將句子和句號組合,過濾空字符串
        result = []
        for i in range(0, len(sentences)-1, 2):
            sentence = sentences[i] + sentences[i+1]
            if sentence.strip():
                result.append(sentence.strip())
                
        # 處理最後一個句子(如果沒有以句號結尾)
        if len(sentences) % 2 == 1 and sentences[-1].strip():
            result.append(sentences[-1].strip())
            
        return result
    
    def process_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        處理DataFrame中的內容
        
        Args:
            df: 包含content欄位的DataFrame
            
        Returns:
            處理後的DataFrame,每個句子為一行
        """
        processed_rows = []
        
        for idx, row in df.iterrows():
            if pd.isna(row['content']):
                self.logger.warning(f"跳過空內容: ID {row['_id']}")
                continue
            
            # 切割內容
            sentences = self.split_content(row['content'])
            
            # 為每個句子創建新行
            for i, sentence in enumerate(sentences):
                new_row = row.copy()
                new_row['content'] = sentence
                new_row['chunk_id'] = f"{row['_id']}_{i}"
                new_row['original_id'] = row['_id']
                new_row['chunk_size'] = len(sentence)
                processed_rows.append(new_row)
        
        # 創建新的DataFrame
        processed_df = pd.DataFrame(processed_rows)
        
        # 輸出處理信息
        self.logger.info(f"處理前文檔數: {len(df)}")
        self.logger.info(f"處理後文檔數: {len(processed_df)}")
        self.logger.info(f"平均chunk大小: {processed_df['chunk_size'].mean():.2f}")
        
        return processed_df

# 使用示例
if __name__ == "__main__":
    # 讀取數據
    df = pd.read_csv('./data/Kinmen_20241205.csv')
    
    # 初始化切割器
    splitter = SimpleSentenceSplitter()
    
    # 處理數據
    processed_df = splitter.process_dataframe(df)
    
    processed_df['author'] = processed_df['author'].fillna('NA')

    
    # 儲存結果
    processed_df.to_csv('./data/kinmen_20241205_split_by_period.csv', index=False)
    print("處理完成!")


2024-12-06 10:14:39,556 - INFO - 處理前文檔數: 668
2024-12-06 10:14:39,556 - INFO - 處理後文檔數: 19667
2024-12-06 10:14:39,558 - INFO - 平均chunk大小: 52.88


處理完成!


#### \n符號進行切割文章

In [10]:
import pandas as pd
import re
from typing import List
import logging

class SimpleParagraphSplitter:
    """
    簡單的段落切割器 - 以換行符號為切割依據
    """
    def __init__(self):
        # 設置logging
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s'
        )
        self.logger = logging.getLogger(__name__)
    
    def split_content(self, text: str) -> List[str]:
        """
        將文本按換行符號切割成段落
        
        Args:
            text: 原始文本
            
        Returns:
            切割後的段落列表
        """
        if pd.isna(text):
            return []
        
        # 使用換行符號切割，過濾空段落
        paragraphs = [p.strip() for p in text.split('\n') if p.strip()]
        
        return paragraphs
    
    def process_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        處理DataFrame中的內容
        
        Args:
            df: 包含content欄位的DataFrame
            
        Returns:
            處理後的DataFrame,每個段落為一行
        """
        processed_rows = []
        
        for idx, row in df.iterrows():
            if pd.isna(row['content']):
                self.logger.warning(f"跳過空內容: ID {row['_id']}")
                continue
            
            # 切割內容
            paragraphs = self.split_content(row['content'])
            
            # 為每個段落創建新行
            for i, paragraph in enumerate(paragraphs):
                new_row = row.copy()
                new_row['content'] = paragraph
                new_row['chunk_id'] = f"{row['_id']}_{i}"
                new_row['original_id'] = row['_id']
                new_row['chunk_size'] = len(paragraph)
                processed_rows.append(new_row)
        
        # 創建新的DataFrame
        processed_df = pd.DataFrame(processed_rows)
        
        # 輸出處理信息
        self.logger.info(f"處理前文檔數: {len(df)}")
        self.logger.info(f"處理後文檔數: {len(processed_df)}")
        self.logger.info(f"平均chunk大小: {processed_df['chunk_size'].mean():.2f}")
        
        return processed_df

# 使用示例
if __name__ == "__main__":
    # 讀取數據
    df = pd.read_csv('./Kinmen_20241205.csv')
    
    # 初始化段落切割器
    splitter = SimpleParagraphSplitter()
    
    # 處理數據
    processed_df = splitter.process_dataframe(df)
    
    # 填補作者欄位的空值
    processed_df['author'] = processed_df['author'].fillna('NA')
    
    # 刪除 content 欄位為空的行之前，印出資料筆數
    print(f"刪除空 content 前的資料筆數: {len(processed_df)}")

    # 清理 content 欄位中的空白或特殊字符
    # Step 1: 移除所有空白字符（包括不可見字符）
    processed_df['content'] = processed_df['content'].str.replace(r"\s+", "", regex=True)

    # Step 2: 移除不可見字符（例如零寬字符）
    processed_df['content'] = processed_df['content'].str.replace(r"[\u200b-\u200d\u2060-\u206f]+", "", regex=True)

    # Step 3: 保留至少包含有效文字的行
    processed_df = processed_df[processed_df['content'].str.contains(r"[a-zA-Z0-9\u4e00-\u9fff]+", regex=True)]

    # 刪除 content 欄位為空的行之後，印出資料筆數
    print(f"刪除空 content 後的資料筆數: {len(processed_df)}")
        
    # 儲存結果
    processed_df.to_csv('./data/kinmen_20241205_split_by_paragraph_cleaned.csv', index=False)
    print("處理完成!")


2024-12-29 22:49:12,383 - INFO - 處理前文檔數: 668
2024-12-29 22:49:12,384 - INFO - 處理後文檔數: 10529
2024-12-29 22:49:12,384 - INFO - 平均chunk大小: 98.08


刪除空 content 前的資料筆數: 10529
刪除空 content 後的資料筆數: 9562
處理完成!


In [12]:
df_1 = pd.read_csv('./data/kinmen_20241205_split_by_paragraph_cleaned.csv')
df_2 = pd.read_csv('./data/kinmen_20241205_split_by_paragraph.csv')
# 找出被刪除的資料筆數
deleted_rows = df_2[~df_2['chunk_id'].isin(df_1['chunk_id'])]

# 儲存被刪除的資料筆數
deleted_rows.to_csv('./data/kinmen_cleaned_data.csv', index=False)

# confirming the csv file in the last time is same as the csv which sending on December,24

In [2]:
# 確認 /data/Kinmen_20242105.csv 和 ./Kinmen_20241205.csv 的資料是否相同
import pandas as pd
df_1 = pd.read_csv('./data/Kinmen_20241205.csv')
df_2 = pd.read_csv('/Users/shuyuhsu/code/Xiemen_wechat_BERTTopic/Kinmen_20241205.csv')

# 檢查兩個DataFrame是否相同
df_1.equals(df_2)


True